# Rerank Demo

This notebook demonstrates a complete retrieval pipeline:
1. **Dense retrieval** using OpenAI embeddings + FAISS (vector search)
2. **Sparse retrieval** using BM25 (keyword-based search)
3. **Hybrid search** combining both via Reciprocal Rank Fusion (RRF)
4. **Reranking** the fused results using either:
   - LLM-based reranking (GPT-4o)
   - Cross-Encoder reranking (`sentence-transformers`)

Reranking improves relevance by re-scoring a small set of retrieved documents with a more powerful (but slower) model.


In [2]:
# Install required packages
# !pip install faiss-cpu langchain langchain-community langchain-openai sentence-transformers python-dotenv


In [ ]:
import sys
from pathlib import Path

# -----------------------------------------------------------------------------
# Setup: add project root to sys.path so we can import shared utilities
# -----------------------------------------------------------------------------
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.env_loader import load_environment
load_environment()

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.retrievers import BaseRetriever
from typing import List, Any
from sentence_transformers import CrossEncoder
from collections import Counter
import math


## Sample Data

A small corpus of sentences about Paris / France.
In production, these would be chunks loaded from PDFs, CSVs, or databases.


In [ ]:
# -----------------------------------------------------------------------------
# Sample document corpus — short sentences about Paris / France.
# In a real RAG system these would be chunks loaded from PDFs, CSVs, etc.
# -----------------------------------------------------------------------------
chunks = [
    "The capital of France is great.",
    "The capital of France is huge.",
    "The capital of France is beautiful.",
    """Have you ever visited Paris? It is a beautiful city where you can eat delicious food and see the Eiffel Tower. 
    I really enjoyed all the cities in france, but its capital with the Eiffel Tower is my favorite city.""", 
    "I really enjoyed my trip to Paris, France. The city is beautiful and the food is delicious. I would love to visit again. Such a great capital city."
]

# Wrap raw strings into LangChain Document objects so they work with retrievers
docs = [Document(page_content=sentence) for sentence in chunks]
docs


## Step 1: Dense Retrieval (Embedding-based Search)

We convert documents into vector embeddings using OpenAI's `text-embedding-3-small` model,
store them in a FAISS index, and retrieve the top-K most similar vectors for a query.

**Limitation:** Dense retrieval is great for semantic similarity but can miss exact keyword matches.

In [ ]:
# -----------------------------------------------------------------------------
# Create embeddings and build an in-memory FAISS vector store.
# -----------------------------------------------------------------------------

# Initialize the OpenAI embedding model (1536-dimensional vectors)
embeddings = OpenAIEmbeddings()

# Build the FAISS index from our sample documents
vectorstore = FAISS.from_documents(docs, embeddings)

# Verify the store is populated by running a similarity search
vectorstore


## Step 2: Sparse Retrieval (BM25)

BM25 is a keyword-based scoring function. It excels at matching exact terms
and is complementary to dense vector search.

We implement a lightweight BM25 from scratch (no external dependency needed) so we can
later fuse dense + sparse results.

In [ ]:
# -----------------------------------------------------------------------------
# BM25 implementation from scratch — based on the pattern in rag_v2.
# BM25 scores documents based on term frequency and inverse document frequency.
# -----------------------------------------------------------------------------
class BM25:
    """Minimal BM25 implementation — no external library needed."""

    def __init__(self, documents: list[str], k1: float = 1.5, b: float = 0.75):
        # k1 controls term frequency saturation; b controls length normalization
        self.k1 = k1
        self.b = b
        # Pre-tokenize the corpus (lowercase + split on whitespace)
        self.corpus = [doc.lower().split() for doc in documents]
        self.N = len(self.corpus)
        self.avgdl = sum(len(d) for d in self.corpus) / self.N
        self.df = self._compute_df()
        self.idf = self._compute_idf()

    def _compute_df(self) -> dict[str, int]:
        # Document frequency: how many docs contain each term?
        df: dict[str, int] = {}
        for doc in self.corpus:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        return df

    def _compute_idf(self) -> dict[str, float]:
        # Inverse document frequency: rare terms get higher weight
        idf = {}
        for term, df in self.df.items():
            idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)
        return idf

    def score(self, query_terms: list[str], doc_index: int) -> float:
        doc = self.corpus[doc_index]
        doc_len = len(doc)
        tf_map = Counter(doc)
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = tf_map.get(term, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += self.idf[term] * numerator / denominator
        return score

    def search(self, query: str, k: int = 5) -> list[tuple[int, float]]:
        # Return top-k (doc_index, score) tuples
        query_terms = query.lower().split()
        scores = [(i, self.score(query_terms, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]


# Initialize BM25 with our sample corpus
bm25 = BM25(chunks)
bm25


## Step 3: Hybrid Search (Dense + Sparse via RRF)

Instead of choosing between dense or sparse retrieval, we run both and fuse the results
using **Reciprocal Rank Fusion (RRF)**.

RRF gives each document a score based on its rank in each individual list:
```
RRF_score(doc) = sum( 1 / (k + rank + 1) )
```

Documents that appear highly in *both* dense and sparse results rise to the top.

In [ ]:
# -----------------------------------------------------------------------------
# Reciprocal Rank Fusion: combine dense and sparse ranked lists.
# -----------------------------------------------------------------------------

def reciprocal_rank_fusion(
    dense_results: list[dict],
    sparse_results: list[tuple[int, float]],
    documents: list[dict],
    rrf_k: int = 60,
    k: int = 5,
) -> list[dict]:
    """Fuse dense and sparse ranked lists via RRF."""
    rrf_scores: dict[str, float] = {}

    # Dense ranks: each doc gets 1/(k + rank + 1)
    for rank, doc in enumerate(dense_results):
        doc_id = doc["id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank + 1)

    # Sparse ranks: map BM25 index -> doc id, then score
    id_list = [d["id"] for d in documents]
    for rank, (doc_idx, _bm25_score) in enumerate(sparse_results):
        doc_id = id_list[doc_idx]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank + 1)

    # Sort by combined RRF score and return top-k
    doc_map = {d["id"]: d for d in documents}
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [
        {**doc_map[doc_id], "rrf_score": score}
        for doc_id, score in fused
        if doc_id in doc_map
    ]


# -----------------------------------------------------------------------------
# Hybrid search pipeline: dense retrieval + BM25 + RRF fusion.
# -----------------------------------------------------------------------------

def hybrid_search(
    query: str,
    vectorstore: FAISS,
    bm25: BM25,
    docs: list[Document],
    k: int = 5,
) -> list[dict]:
    """Run dense + sparse search, then fuse with RRF."""
    # 1. Dense retrieval: get top-2k from FAISS
    dense_docs = vectorstore.similarity_search(query, k=k * 2)
    dense_results = [{"id": str(i), "text": d.page_content} for i, d in enumerate(dense_docs)]

    # 2. Sparse retrieval: get top-2k from BM25
    sparse_results = bm25.search(query, k=k * 2)

    # 3. Fuse via RRF
    documents = [{"id": str(i), "text": d.page_content} for i, d in enumerate(docs)]
    return reciprocal_rank_fusion(dense_results, sparse_results, documents, k=k)


# -----------------------------------------------------------------------------
# Execute hybrid search
# -----------------------------------------------------------------------------

query = "what is the capital of france?"

# Run the hybrid search pipeline
hybrid_results = hybrid_search(query, vectorstore, bm25, docs, k=5)

print("Hybrid search results (dense + sparse fusion):")
for i, res in enumerate(hybrid_results):
    print(f"\nResult {i+1}:")
    print(res["text"])
    print(f"  RRF score: {res["rrf_score"]:.4f}")


## Method 1: LLM-based Reranking

Now we take the hybrid search results and re-score them using GPT-4o.
The LLM acts as a cross-encoder: it sees the query and document together,
so it can judge true relevance beyond keyword overlap or vector similarity.

**Trade-off:** slower and more expensive than a neural cross-encoder,
but leverages the LLM's reasoning ability.

In [ ]:
# -----------------------------------------------------------------------------
# LLM-based reranker: ask GPT to score each candidate's relevance (0-10).
# -----------------------------------------------------------------------------

class RatingScore(BaseModel):
    relevance_score: float = Field(..., description="The relevance score of a document to a query.")


def rerank_documents_llm(
    query: str,
    docs: list[dict],
    top_n: int = 3,
    model: str = "gpt-4o",
) -> list[dict]:
    """Use an LLM to score and rerank documents based on relevance to the query."""
    prompt_template = PromptTemplate(
        input_variables=["query", "doc"],
        template="""On a scale of 1-10, rate the relevance of the following document to the query.\n\n'
        Query: {query}
        Document: {doc}
        Relevance Score:"""
    )
    llm = ChatOpenAI(temperature=0, model_name=model)
    llm_chain = prompt_template | llm.with_structured_output(RatingScore)

    scored_docs = []
    for doc in docs:
        score = llm_chain.invoke({"query": query, "doc": doc["text"]}).relevance_score
        try:
            score = float(score)
        except ValueError:
            score = 0.0
        scored_docs.append((doc, score))

    # Sort by score descending and keep top_n
    reranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]


# -----------------------------------------------------------------------------
# Run LLM reranking on hybrid results
# -----------------------------------------------------------------------------

llm_reranked = rerank_documents_llm(query, hybrid_results, top_n=3)

print("Top initial documents (hybrid):")
for i, doc in enumerate(hybrid_results[:3]):
    print(f"\nDocument {i+1}:")
    print(doc["text"])
    print(f"  RRF score: {doc["rrf_score"]:.4f}")

print(f"\nQuery: {query}\n")
print("Top reranked documents (LLM):")
for i, doc in enumerate(llm_reranked):
    print(f"\nDocument {i+1}:")
    print(doc["text"])


## Method 2: Cross-Encoder Reranking

A dedicated reranking model (`sentence-transformers` cross-encoder) is typically
faster and cheaper than an LLM reranker, and still much more accurate than
initial retrieval alone.

We wrap it in a LangChain `BaseRetriever` so it fits cleanly into retrieval pipelines.

In [ ]:
# -----------------------------------------------------------------------------
# Cross-Encoder reranker using sentence-transformers.
# This model is specifically trained for passage reranking (MS MARCO dataset).
# -----------------------------------------------------------------------------
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


class CrossEncoderRetriever(BaseRetriever, BaseModel):
    vectorstore: Any = Field(description="Vector store for initial retrieval")
    cross_encoder: Any = Field(description="Cross-encoder model for reranking")
    k: int = Field(default=5, description="Number of documents to retrieve initially")
    rerank_top_k: int = Field(default=3, description="Number of documents to return after reranking")

    class Config:
        arbitrary_types_allowed = True

    def get_relevant_documents(self, query: str) -> List[dict]:
        # 1. Retrieve a broad set from the vector store
        initial_docs = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Score each (query, document) pair with the cross-encoder
        pairs = [[query, doc.page_content] for doc in initial_docs]
        scores = self.cross_encoder.predict(pairs)

        # 3. Sort by cross-encoder score and return top-k
        scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

    async def aget_relevant_documents(self, query: str) -> List[dict]:
        raise NotImplementedError("Async retrieval not implemented")


# -----------------------------------------------------------------------------
# Instantiate the retriever and run it
# -----------------------------------------------------------------------------

cross_encoder_retriever = CrossEncoderRetriever(
    vectorstore=vectorstore,
    cross_encoder=cross_encoder,
    k=10,
    rerank_top_k=5
)

ce_docs = cross_encoder_retriever.get_relevant_documents(query)
print("Cross-encoder reranked documents:")
for i, doc in enumerate(ce_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)


## Comparison: Baseline vs LLM Rerank vs Cross-Encoder

We compare three retrieval strategies side-by-side:
1. **Baseline** — raw FAISS similarity search (dense only)
2. **LLM rerank** — hybrid search + GPT-4o reranking
3. **Cross-Encoder** — hybrid search + cross-encoder reranking

Reranking typically moves the most relevant documents to the top, even when
the initial retrieval missed them.

In [ ]:
print("Baseline vector search (dense only):")
for i, doc in enumerate(vectorstore.similarity_search(query, k=3)):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

print("\n\nAfter LLM reranking:")
for i, doc in enumerate(llm_reranked):
    print(f"\nDocument {i+1}:")
    print(doc["text"])

print("\n\nAfter Cross-Encoder reranking:")
for i, doc in enumerate(ce_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)
